# Cocogoat 2026 - Task 1: modern GAN at 64x64

R3GAN objective (RpGAN + R1 + R2) with normalization-free depthwise-separable blocks and self-attention, trained on your Genshin fan-art set (`Complete_dataset.npy`, 45,935 images) with CLIP k-means pseudo-classes (cluster-conditioned GAN) against mode collapse. Budget: ~3.5 h of training + ~10 min data prep + final eval. Outputs: `MyDrive/Cocogoat/runs/task1/`.

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, set the budgets in the settings cell (5),
then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) GPU smoke test (tiny model, ~10 steps, always from scratch). The first run also converts the 11 GB
#    float64 file into a 1.4 GB uint8 cache (MyDrive/Cocogoat/cache) - a few minutes, reused by every later run.
!python -m cocogoat.tasks.task1_gan64 --smoke --dataset genshin --run_dir /content/drive/MyDrive/Cocogoat/runs/task1_smoke --force

In [ ]:
# 5) Settings - edit freely. HOURS = training time budget (evaluation/data time not included).
#    Raising HOURS later on the same RUN_DIR continues training from the last checkpoint.
#    The first 3.5 h run (runs/task1) diverged after ~2.4 h; the fixed recipe (gamma floor 1.0, divergence guard,
#    evaluation every 20 min) should start in a NEW folder, otherwise it would resume from the diverged weights.
HOURS = 3.5
RUN_DIR = '/content/drive/MyDrive/Cocogoat/runs/task1_v2'

In [ ]:
# 6) Training (resumes automatically after a disconnect). At the end both the final model and the best
#    checkpoint seen during training are evaluated on 50k samples (G_ema_final.pt / G_ema_best.pt).
!python -m cocogoat.tasks.task1_gan64 --dataset genshin --hours {HOURS} --run_dir {RUN_DIR}

In [ ]:
# 6b) Optional, no training: evaluate the best checkpoint of an OLD run on 50k samples
#     (writes G_ema_best.pt, samples/best.png, samples/best_per_class.png and best_metrics.json there).
OLD_RUN = '/content/drive/MyDrive/Cocogoat/runs/task1'
EVAL_OLD_BEST = False
if EVAL_OLD_BEST:
    !python -m cocogoat.tasks.task1_gan64 --dataset genshin --eval_best --run_dir {OLD_RUN}

In [ ]:
# 7) Results
import json
from IPython.display import Image, Markdown, display
run = RUN_DIR
display(Markdown(open(f'{run}/report.md').read()))
for name in ['final.png', 'final_per_class.png', 'best.png', 'best_per_class.png']:
    try:
        display(Markdown(f'**{name}**'), Image(f'{run}/samples/{name}'))
    except FileNotFoundError:
        pass